# 03 — Patient-Level Train / Val / Test Split

 **Purpose:** Assign each patient to exactly one split (train / val / test) and
 save per-split manifests.  All images of a patient move together — no leakage.

 **This notebook does NOT:** resize images, generate masks, or train a model.

# 1  Imports

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

print(f"pandas   {pd.__version__}")
print(f"numpy    {np.__version__}")

pandas   2.3.3
numpy    2.4.4


# 2  Constants

In [2]:
OUTPUT_ROOT          = Path(r"D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs")
MASTER_MASK_MANIFEST = OUTPUT_ROOT / "01_manifest" / "segmentation_master_manifest_with_masks.csv"
SPLITS_DIR           = OUTPUT_ROOT / "03_splits"

SEED        = 42
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Split ratios must sum to 1.0"
assert MASTER_MASK_MANIFEST.exists(), (
    f"Master mask manifest not found: {MASTER_MASK_MANIFEST}\n"
    "Run Notebooks 1 and 2 first."
)
print(f"Manifest found: {MASTER_MASK_MANIFEST}")

Manifest found: D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\01_manifest\segmentation_master_manifest_with_masks.csv


# 3  Create Output Folder


In [3]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Ready: {SPLITS_DIR}")

Ready: D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\03_splits


# 4  Load Manifest

In [4]:
df_raw = pd.read_csv(MASTER_MASK_MANIFEST)
print(f"Raw manifest rows : {len(df_raw)}")
print(f"Columns           : {list(df_raw.columns)}")

Raw manifest rows : 5265
Columns           : ['sample_id', 'patient_id', 'image_id', 'image_path', 'json_path', 'segmented_image_path', 'image_filename', 'json_filename', 'segmented_image_filename', 'image_width', 'image_height', 'json_exists', 'segmented_image_exists', 'tongue_shape_count', 'has_valid_tongue_polygon', 'usable_for_mask_generation', 'exclusion_reason', 'mask_path', 'mask_created', 'mask_width', 'mask_height', 'mask_unique_values', 'mask_foreground_pixels', 'mask_foreground_ratio', 'mask_valid_binary', 'mask_ratio_warning', 'mask_error']


# 5  Validate Required Columns

In [5]:
REQUIRED_COLUMNS = [
    "sample_id", "patient_id", "image_id",
    "image_path", "json_path", "segmented_image_path", "mask_path",
    "image_width", "image_height",
    "usable_for_mask_generation", "mask_created", "mask_valid_binary",
]

missing_cols = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]
assert len(missing_cols) == 0, (
    f"Required columns missing from manifest: {missing_cols}\n"
    "Ensure Notebooks 1 and 2 completed correctly."
)
print("All required columns present.")


All required columns present.


# 6  Filter to Fully Valid Rows

In [6]:
df = df_raw[
    (df_raw["usable_for_mask_generation"] == True) &
    (df_raw["mask_created"]               == True) &
    (df_raw["mask_valid_binary"]          == True)
].copy()

print(f"Rows after filtering (fully valid) : {len(df)}")
print(f"Rows excluded                      : {len(df_raw) - len(df)}")

assert len(df) > 0, (
    "No fully valid rows remain after filtering. "
    "Check mask generation results in Notebook 2."
)

Rows after filtering (fully valid) : 5265
Rows excluded                      : 0


# 7  Validate patient_id and sample_id Integrity

In [7]:
null_pid = df["patient_id"].isnull().sum()
assert null_pid == 0, f"{null_pid} rows have null patient_id — cannot split safely."

dup_sid = df["sample_id"].duplicated().sum()
assert dup_sid == 0, f"{dup_sid} duplicate sample_ids found — manifest is corrupted."

# Ensure patient_id is string for consistent handling
df["patient_id"] = df["patient_id"].astype(str)

print(f"Unique patients : {df['patient_id'].nunique()}")
print(f"Total samples   : {len(df)}")
print("patient_id and sample_id integrity checks passed.")

Unique patients : 4641
Total samples   : 5265
patient_id and sample_id integrity checks passed.


# 8  Patient-Level Split
We split the list of unique patient IDs, not individual images.
All images belonging to a patient move to the same split.
This prevents any form of patient-level data leakage between splits.

In [8]:
unique_patients = sorted(df["patient_id"].unique())
n_patients      = len(unique_patients)

assert n_patients >= 7, (
    f"Only {n_patients} unique patients found. Need at least 7 for a 70/15/15 split "
    "to have at least 1 patient in val and test."
)

# Step 1: split off test set
val_test_ratio = VAL_RATIO + TEST_RATIO   # fraction of total going to val+test
patients_train, patients_val_test = train_test_split(
    unique_patients,
    test_size=val_test_ratio,
    random_state=SEED,
    shuffle=True,
)

# Step 2: split val_test into val and test
# Within val+test pool, test fraction = TEST_RATIO / (VAL_RATIO + TEST_RATIO)
test_fraction_of_val_test = TEST_RATIO / val_test_ratio
patients_val, patients_test = train_test_split(
    patients_val_test,
    test_size=test_fraction_of_val_test,
    random_state=SEED,
    shuffle=True,
)

print(f"Train patients : {len(patients_train)}")
print(f"Val   patients : {len(patients_val)}")
print(f"Test  patients : {len(patients_test)}")

Train patients : 3248
Val   patients : 696
Test  patients : 697


# 9  Assert No Patient Appears in More Than One Split

In [9]:
set_train = set(patients_train)
set_val   = set(patients_val)
set_test  = set(patients_test)

assert set_train.isdisjoint(set_val),  "LEAKAGE: patient(s) appear in both train and val."
assert set_train.isdisjoint(set_test), "LEAKAGE: patient(s) appear in both train and test."
assert set_val.isdisjoint(set_test),   "LEAKAGE: patient(s) appear in both val and test."
assert set_train | set_val | set_test == set(unique_patients), (
    "Not all patients are assigned to a split."
)
print("No-leakage assertions passed — all patients assigned to exactly one split.")

No-leakage assertions passed — all patients assigned to exactly one split.


# 10  Assign Split Labels to DataFrame

In [10]:
pid_to_split = {}
for p in patients_train: pid_to_split[p] = "train"
for p in patients_val:   pid_to_split[p] = "val"
for p in patients_test:  pid_to_split[p] = "test"

df["split_group_id"]   = df["patient_id"]
df["split_assignment"] = df["patient_id"].map(pid_to_split)

assert df["split_assignment"].isnull().sum() == 0, (
    "Some rows could not be assigned a split — check patient_id mapping."
)
print("Split assignment complete.")
print(df["split_assignment"].value_counts().to_string())


Split assignment complete.
split_assignment
train    3671
test      804
val       790


# 11  Save Split Manifests

In [11]:
df_train = df[df["split_assignment"] == "train"].copy()
df_val   = df[df["split_assignment"] == "val"].copy()
df_test  = df[df["split_assignment"] == "test"].copy()

train_path   = SPLITS_DIR / "segmentation_train_manifest.csv"
val_path     = SPLITS_DIR / "segmentation_val_manifest.csv"
test_path    = SPLITS_DIR / "segmentation_test_manifest.csv"
full_path    = SPLITS_DIR / "segmentation_manifest_with_splits.csv"
summary_path = SPLITS_DIR / "segmentation_split_summary.csv"

df_train.to_csv(train_path, index=False)
df_val.to_csv(val_path,     index=False)
df_test.to_csv(test_path,   index=False)
df.to_csv(full_path,        index=False)

print(f"Train manifest saved -> {train_path}")
print(f"Val   manifest saved -> {val_path}")
print(f"Test  manifest saved -> {test_path}")
print(f"Full  manifest saved -> {full_path}")


Train manifest saved -> D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\03_splits\segmentation_train_manifest.csv
Val   manifest saved -> D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\03_splits\segmentation_val_manifest.csv
Test  manifest saved -> D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\03_splits\segmentation_test_manifest.csv
Full  manifest saved -> D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\03_splits\segmentation_manifest_with_splits.csv


# 12  Build and Save Split Summary CSV

In [12]:
summary_rows = []
for split_name, split_df, split_patients in [
    ("train", df_train, patients_train),
    ("val",   df_val,   patients_val),
    ("test",  df_test,  patients_test),
]:
    imgs_per_patient = split_df.groupby("patient_id").size()
    summary_rows.append({
        "split"              : split_name,
        "n_samples"          : len(split_df),
        "n_patients"         : len(split_patients),
        "pct_samples"        : round(len(split_df) / len(df) * 100, 2),
        "pct_patients"       : round(len(split_patients) / n_patients * 100, 2),
        "min_imgs_per_patient": imgs_per_patient.min(),
        "max_imgs_per_patient": imgs_per_patient.max(),
        "mean_imgs_per_patient": round(imgs_per_patient.mean(), 2),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(summary_path, index=False)
print(f"Summary saved -> {summary_path}")


Summary saved -> D:\DIABETES\Segmentation_Dataset\Segmentation_Branch_Outputs\03_splits\segmentation_split_summary.csv


# 13  Print Full Summary

In [13]:
print("=" * 60)
print(f"  Total samples (valid)         : {len(df)}")
print(f"  Total unique patients         : {n_patients}")
print("-" * 60)
for _, row in summary_df.iterrows():
    print(f"  [{row['split'].upper():5s}]  "
          f"samples={row['n_samples']:4d} ({row['pct_samples']:5.1f}%)  |  "
          f"patients={row['n_patients']:3d} ({row['pct_patients']:5.1f}%)  |  "
          f"imgs/patient: min={row['min_imgs_per_patient']} "
          f"max={row['max_imgs_per_patient']} "
          f"mean={row['mean_imgs_per_patient']:.2f}")
print("=" * 60)

  Total samples (valid)         : 5265
  Total unique patients         : 4641
------------------------------------------------------------
  [TRAIN]  samples=3671 ( 69.7%)  |  patients=3248 ( 70.0%)  |  imgs/patient: min=1 max=3 mean=1.13
  [VAL  ]  samples= 790 ( 15.0%)  |  patients=696 ( 15.0%)  |  imgs/patient: min=1 max=4 mean=1.14
  [TEST ]  samples= 804 ( 15.3%)  |  patients=697 ( 15.0%)  |  imgs/patient: min=1 max=3 mean=1.15


# Notebook Complete

 | Output | Location |
 |--------|----------|
 | Train manifest | `03_splits/segmentation_train_manifest.csv` |
 | Val manifest   | `03_splits/segmentation_val_manifest.csv` |
 | Test manifest  | `03_splits/segmentation_test_manifest.csv` |
 | Full manifest with splits | `03_splits/segmentation_manifest_with_splits.csv` |
 | Split summary  | `03_splits/segmentation_split_summary.csv` |

 **No patient appears in more than one split.**
 **Source folders were not modified.**
 **Next step:** run `04_pytorch_dataset_preprocessing.ipynb`.